Step-1. Data Preprocessing
1. Load the Dataset

In [6]:
import pandas as pd

# Load the dataset
df = pd.read_csv('malware_dataset.csv')

# Check for missing values
print(df.isnull().sum())

# Drop missing values if any
df.dropna(inplace=True)

# View dataset structure
print(df.head())


asm_commands_add     0
asm_commands_call    0
asm_commands_cdq     0
asm_commands_cld     0
asm_commands_cli     0
                    ..
asm_commands_xchg    0
asm_commands_xor     0
line_count_asm       0
size_asm             0
Class                0
Length: 69, dtype: int64
   asm_commands_add  asm_commands_call  asm_commands_cdq  asm_commands_cld  \
0               436              646.0               0.0              10.0   
1               469              262.0               0.0               4.0   
2              1587             1828.0               0.0               0.0   
3               213              227.0               0.0               0.0   
4                36               76.0               0.0               0.0   

   asm_commands_cli  asm_commands_cmc  asm_commands_cmp  asm_commands_cwd  \
0               9.0               0.0             228.0               0.0   
1               1.0               4.0             185.0               4.0   
2              31.0   

2. Encode Labels (Benign vs. Malware)

In [7]:
from sklearn.preprocessing import LabelEncoder

# Check the actual column names
print("Columns in dataset:", df.columns)

# If the column name is different, rename it
if 'Label' in df.columns:
    df.rename(columns={'Label': 'label'}, inplace=True)
elif 'Class' in df.columns:  # Some datasets use 'Class' instead of 'Label'
    df.rename(columns={'Class': 'label'}, inplace=True)

# Ensure 'label' column exists
if 'label' in df.columns:
    # Remove missing values in 'label' column
    df = df.dropna(subset=['label'])

    # Encode labels
    encoder = LabelEncoder()
    df['label'] = encoder.fit_transform(df['label'])
    print("Encoding successful!")
else:
    print("Error: 'label' column not found in dataset.")


Columns in dataset: Index(['asm_commands_add', 'asm_commands_call', 'asm_commands_cdq',
       'asm_commands_cld', 'asm_commands_cli', 'asm_commands_cmc',
       'asm_commands_cmp', 'asm_commands_cwd', 'asm_commands_daa',
       'asm_commands_dd', 'asm_commands_dec', 'asm_commands_dw',
       'asm_commands_endp', 'asm_commands_faddp', 'asm_commands_fchs',
       'asm_commands_fdiv', 'asm_commands_fdivr', 'asm_commands_fistp',
       'asm_commands_fld', 'asm_commands_fstp', 'asm_commands_fword',
       'asm_commands_fxch', 'asm_commands_imul', 'asm_commands_in',
       'asm_commands_inc', 'asm_commands_ins', 'asm_commands_jb',
       'asm_commands_je', 'asm_commands_jg', 'asm_commands_jl',
       'asm_commands_jmp', 'asm_commands_jnb', 'asm_commands_jno',
       'asm_commands_jo', 'asm_commands_jz', 'asm_commands_lea',
       'asm_commands_mov', 'asm_commands_mul', 'asm_commands_not',
       'asm_commands_or', 'asm_commands_out', 'asm_commands_outs',
       'asm_commands_pop', 'asm_comm

Step-2. Feature Extraction
Static Analysis (Extract PE Headers, Opcodes, File Size, and Hashes)

In [8]:
import pefile
import hashlib

def extract_static_features(file_path):
    pe = pefile.PE(file_path)
    features = {
        'file_size': pe.OPTIONAL_HEADER.SizeOfImage,
        'entry_point': pe.OPTIONAL_HEADER.AddressOfEntryPoint,
        'checksum': hashlib.md5(open(file_path,'rb').read()).hexdigest()
    }
    return features


Dynamic Analysis (Monitor API Calls, System Behavior, and Network Activity)

In [9]:
import os

def extract_dynamic_features(file_path):
    os.system(f"cuckoo submit {file_path}")


Model Development and Training
Split Data for Training and Testing

In [10]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=['label'])  # Features
y = df['label']  # Labels

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


1. Train a Random Forest Classifier (Baseline Model)

In [11]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)

y_pred_rf = rf_model.predict(X_test)
print("Random Forest Accuracy:", accuracy_score(y_test, y_pred_rf))


Random Forest Accuracy: 0.9875804967801288


2. Train a Deep Learning Model (CNN)

In [12]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, Flatten, Dense

model = Sequential([
    Conv1D(filters=32, kernel_size=3, activation='relu', input_shape=(X_train.shape[1], 1)),
    Flatten(),
    Dense(64, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
model.fit(X_train, y_train, epochs=10, batch_size=32, validation_data=(X_test, y_test))


ModuleNotFoundError: No module named 'tensorflow'

Step 6: Model Evaluation

In [16]:
from sklearn.metrics import classification_report

y_pred_dl = model.predict(X_test) > 0.5  # Convert probabilities to binary labels
print("Deep Learning Model Performance:\n", classification_report(y_test, y_pred_dl))


NameError: name 'model' is not defined

Step 7: Deployment (Real-Time Malware Classification)

In [17]:
def predict_malware(file_path):
    features = extract_static_features(file_path)
    features_df = pd.DataFrame([features])
    prediction = rf_model.predict(features_df)
    return "Malware" if prediction[0] == 1 else "Benign"

print(predict_malware("sample.exe"))
